# Rabbit — Colab smoke test

This notebook clones [Stanley03-arch/Rabbit](https://github.com/Stanley03-arch/Rabbit), builds the C engine, creates a synthetic model, and runs a non-interactive generation pass.

**Goal:** confirm the pipeline works in a clean Linux environment (Colab).

Open in Colab: use *File → Upload notebook* or the GitHub badge link below after pushing.

## 1. System info

In [ ]:
!uname -a
!gcc --version | head -1
!python3 --version
!free -h | head -2

## 2. Clone repository

In [ ]:
import os
os.chdir("/content")
!rm -rf Rabbit
!git clone --depth 1 https://github.com/Stanley03-arch/Rabbit.git
os.chdir("/content/Rabbit")
!git log -1 --oneline
!ls -la

## 3. Build engine

In [ ]:
!make clean
!make 2>&1
!ls -la rabbit
!./rabbit version

## 4. Create synthetic test model

In [ ]:
!make test-model 2>&1
!ls -la models/

## 5. Non-interactive chat smoke test

Feeds a prompt and `quit` so the session exits cleanly. You should see model info, generated bytes, and cache statistics.

In [ ]:
!printf 'hello rabbit\nquit\n' | ./rabbit chat --model models/tiny-test.rabbit --tokens 24 --temp 0.7

## 6. Optional: larger int4 demo model

Requires `numpy` (preinstalled on Colab).

In [ ]:
!python3 tools/convert_hf_to_rabbit.py --demo --out models/demo-moe.rabbit --hidden-dim 128 --n-experts 16 --quant 4
!ls -lh models/demo-moe.rabbit
!printf 'demo prompt\nquit\n' | ./rabbit chat --model models/demo-moe.rabbit --tokens 32 --temp 0.8

## 7. Success checklist

Mark each item after the cells above run:

| Step | Expected |
|------|----------|
| Build | `./rabbit version` prints `Rabbit 0.2.x` |
| Test model | `models/tiny-test.rabbit` exists |
| Smoke chat | Model table printed, some output tokens, then cache hits/misses |
| Demo (optional) | int4 demo loads and generates |

If the smoke chat shows **expert_map: mapped** and a **cache hit rate**, the streaming path is working.

Generated text from synthetic weights will look random — that is expected.

## Troubleshooting

- **Permission denied on `./rabbit`:** run `!chmod +x rabbit tools/create_test_model`
- **`make: gcc: not found`:** Colab normally has gcc; try `!apt-get install -y build-essential`
- **Converter import error:** `!pip install -q numpy`

Repo: https://github.com/Stanley03-arch/Rabbit